# ML-06 — Validation & Leakage Audit

**Lane:** CTR / Engagement Opportunity Scoring

**Purpose:** audit the Week-5 Decision Tree like a research reviewer would audit a published result.

This notebook does four things:

1. reviews two findings from the FlyRank research paper and asks constructive methodology questions;
2. compares a naive random split with the honest time-aware split used for the forward-looking task;
3. audits every model feature for leakage;
4. inspects real failure examples and rewrites claims so they stay at the level of **observed, measured, directional, and decision-support** evidence.

The goal is not to make the model look better. The goal is to make the evidence harder to fool.

## 1. Two paper findings + my methodology questions

### Finding A — Click Capture by Position Tier

The research paper reports weighted CTR of **0.420% for Top 3, 0.340% for positions 4–10, 0.325% for positions 11–20, 0.163% for positions 21–50, and 0.050% for positions 50+**. The paper interprets this as a steep decline in click capture as position worsens.

**Methodology question:** What exactly is the comparison unit and denominator behind each weighted CTR? Because CTR is clicks divided by impressions, the weighting choice can materially change the result. I would want to confirm that the same search-performance window and inclusion rules are used across tiers and that the pattern is not being driven by a small number of very high-volume pages or clients.

**Validation question:** Because this is a descriptive portfolio finding rather than a prediction model, I would want to see whether the direction of the pattern remains similar across separate time windows or client groups. That would strengthen the claim from one observed aggregate pattern to a more stable directional finding.

This is a constructive review question, not a claim that the paper's calculation is wrong.

### Finding B — Reader Engagement and Search Visibility Move Together

The paper reports that **high scroll + high reader engagement is associated with +16.1 health points** and recommends tracking engagement measures by channel.

**Methodology question:** I would first check how much of the reported relationship is mechanically shared with the definition of the Health Score. The paper defines Health Score using impressions, position, CTR and scroll depth. If scroll depth is itself an ingredient of the outcome being compared, part of the relationship can be definitional rather than independent evidence.

**Validation question:** I would ask whether the same relationship appears when the outcome is changed to a measure that does not contain scroll depth or engagement as one of its ingredients, and whether it persists across time/client groups. That would help distinguish a descriptive association from a more generalizable signal.

Again, the purpose is to sharpen the evidence standard rather than to reject the finding.

### Source note

The paper explicitly says its headline findings prioritize direct aggregate comparisons, while its ML pages are exploratory appendix material. It also states that the study is a pattern study rather than proof of cause and effect. This audit follows that same standard.

In [4]:
# Source / scope notes used for this audit
PAPER_FINDINGS = {
    "position_ctr": "Weighted CTR by position tier: Top 3 0.420%, Page 1 0.340%, Striking Distance 0.325%, Page 3-5 0.163%, Deep 0.050%.",
    "engagement": "High scroll + high reader engagement was reported as associated with +16.1 health points."
}

print(PAPER_FINDINGS["position_ctr"])
print(PAPER_FINDINGS["engagement"])

Weighted CTR by position tier: Top 3 0.420%, Page 1 0.340%, Striking Distance 0.325%, Page 3-5 0.163%, Deep 0.050%.
High scroll + high reader engagement was reported as associated with +16.1 health points.


## 2. My model under an honest split — before / after

### Why the split needs an audit

The Week-5 task predicts a future-month CTR drop from information available at month *t*. A random row split can mix different months and therefore does not reproduce the decision that the model is supposed to support.

For a forward-looking task, the honest design is:

- **Train:** March → April
- **Validation:** April → May
- **Test:** May → June

The comparison below intentionally shows the naive random result first, then the time-aware result. The point of the before/after is to expose how validation design can change the apparent strength of a model.

In [5]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN secret not found in Colab.")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

print("Warehouse connection ready.")

Warehouse connection ready.


In [6]:
# Aggregate the warehouse to one row per client/content/month.
monthly = con.sql(f'''
SELECT
    client_hash_id,
    content_hash_id,
    month,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
        ELSE 0.0
    END AS ctr,
    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS avg_position,
    SUM(ga4_sessions) AS ga4_sessions,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
    SUM(scroll_events) AS scroll_events
FROM {FACT}
WHERE month IN ('2026-03','2026-04','2026-05','2026-06')
GROUP BY client_hash_id, content_hash_id, month
''').df()

monthly["engagement_rate"] = np.where(
    monthly["ga4_sessions"] > 0,
    monthly["ga4_engaged_sessions"] / monthly["ga4_sessions"],
    np.nan
)

print("Monthly rows:", len(monthly))
display(monthly.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Monthly rows: 1491967


,client_hash_id,content_hash_id,month,impressions,clicks,ctr,avg_position,ga4_sessions,ga4_engaged_sessions,scroll_events,engagement_rate
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,2026-03,181.0,0.0,0.000000,5.171271,NaN,NaN,NaN,NaN
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,2026-03,46.0,1.0,0.021739,4.543478,NaN,NaN,NaN,NaN
2,client_62f4a7e64f5e0096,content_ac8663da7484669a,2026-03,34.0,0.0,0.000000,5.941176,NaN,NaN,NaN,NaN
3,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,2026-03,3108.0,0.0,0.000000,6.953668,NaN,NaN,NaN,NaN
4,client_62f4a7e64f5e0096,content_d49a012dcb924e31,2026-03,329.0,0.0,0.000000,5.136778,NaN,NaN,NaN,NaN


In [7]:
def make_pair(current_month, future_month):
    cur = monthly[monthly["month"] == current_month].copy()
    fut = monthly[monthly["month"] == future_month][
        ["client_hash_id", "content_hash_id", "ctr"]
    ].rename(columns={"ctr": "future_ctr"})

    out = cur.merge(
        fut,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )

    # Minimum current visibility keeps tiny denominators from dominating.
    out = out[out["impressions"] >= 50].copy()

    # Future label: used only after features are fixed.
    out["target"] = (out["future_ctr"] < out["ctr"]).astype(int)

    out["position_bucket"] = pd.cut(
        out["avg_position"],
        bins=[-np.inf, 3, 10, 20, 50, np.inf],
        labels=["1-3", "4-10", "11-20", "21-50", "51+"]
    )
    return out

train = make_pair("2026-03", "2026-04")
valid = make_pair("2026-04", "2026-05")
test = make_pair("2026-05", "2026-06")

print("Train:", train.shape)
print("Valid:", valid.shape)
print("Test :", test.shape)

Train: (116114, 14)
Valid: (125758, 14)
Test : (133719, 14)


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

FEATURES = [
    "impressions",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_events"
]

def prepare_X(df):
    x = df[FEATURES].replace([np.inf, -np.inf], np.nan).copy()
    return x.fillna(x.median(numeric_only=True))

def precision_at_k(y, score, k=50):
    order = np.argsort(-np.asarray(score))
    top = np.asarray(y)[order[:min(k, len(order))]]
    return float(top.mean()) if len(top) else np.nan

# ---- BEFORE: naive random row split ----
all_rows = pd.concat([train, valid], ignore_index=True)
X_all = prepare_X(all_rows)
y_all = all_rows["target"]

X_random_train, X_random_test, y_random_train, y_random_test = train_test_split(
    X_all,
    y_all,
    test_size=0.20,
    random_state=42,
    stratify=y_all
)

random_model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=50,
    random_state=42
)
random_model.fit(X_random_train, y_random_train)

random_score = random_model.predict_proba(X_random_test)[:, 1]
random_p50 = precision_at_k(y_random_test, random_score, 50)

# ---- AFTER: honest time-aware split ----
X_train = prepare_X(train)
y_train = train["target"]

X_test = prepare_X(test)
y_test = test["target"]

time_model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=50,
    random_state=42
)
time_model.fit(X_train, y_train)

time_score = time_model.predict_proba(X_test)[:, 1]
time_p50 = precision_at_k(y_test, time_score, 50)

comparison = pd.DataFrame([
    {
        "validation_design": "Random 80/20 (before)",
        "evaluation_window": "Mixed March+April rows",
        "Precision@50": random_p50
    },
    {
        "validation_design": "Time-aware (after)",
        "evaluation_window": "May -> June",
        "Precision@50": time_p50
    }
])

display(comparison)

,validation_design,evaluation_window,Precision@50
0,Random 80/20 (before),Mixed March+April rows,0.88
1,Time-aware (after),May -> June,0.82


### Before / after interpretation

The random split is useful only as a diagnostic. It is not the preferred estimate for a forward-looking deployment decision because rows from the same months are mixed across train and test.

The time-aware result is the evidence to use when describing this model's forward-looking performance. If the score falls under the honest split, that is not a failure of the audit — it is evidence that the random estimate was optimistic.

In [9]:
print("Random-split Precision@50:", round(random_p50, 4))
print("Time-aware Precision@50:", round(time_p50, 4))
print("Difference (random - time-aware):", round(random_p50 - time_p50, 4))

Random-split Precision@50: 0.88
Time-aware Precision@50: 0.82
Difference (random - time-aware): 0.06


## 3. Leakage audit

The audit asks whether any model feature contains the future answer or a product decision that was derived from the answer.

### Feature-by-feature decision

| Feature | Decision-time safe? | Reason |
|---|---|---|
| `impressions` | YES | measured in the current month |
| `ctr` | YES | calculated from current-month clicks/impressions |
| `avg_position` | YES | current-month search visibility |
| `engagement_rate` | YES | calculated from current-month GA4 sessions/engaged sessions |
| `scroll_events` | YES | current-month observed engagement |
| `future_ctr` | NO — label only | measured in the following month |
| `target` | NO — label only | derived from future CTR |
| product flags / scores | NO | should never be used as ordinary model features |

The key rule is that the future window is used to construct the evaluation label, never the feature matrix.

In [10]:
# Programmatic leakage checks
forbidden_feature_names = {
    "future_ctr",
    "target",
    "future_impressions",
    "future_clicks",
    "future_position",
    "priority_score",
    "action_type",
    "health_score",
    "refresh_tier",
    "needs_ctr_fix",
    "is_quick_win"
}

feature_check = pd.DataFrame({
    "feature": FEATURES,
    "contains_future_or_product_name": [
        f.lower() in forbidden_feature_names
        or any(token in f.lower() for token in ["future_", "priority", "action_type", "health_score", "refresh_tier"])
        for f in FEATURES
    ]
})

display(feature_check)

assert not feature_check["contains_future_or_product_name"].any(), (
    "Leakage audit failed: a forbidden feature name is present."
)

# Confirm that future_ctr and target are not in the model matrix.
assert "future_ctr" not in FEATURES
assert "target" not in FEATURES

print("LEAKAGE AUDIT: PASS")

,feature,contains_future_or_product_name
0,impressions,False
1,ctr,False
2,avg_position,False
3,engagement_rate,False
4,scroll_events,False


LEAKAGE AUDIT: PASS


In [11]:
# Check that the feature columns exist in the current warehouse-derived frame.
missing = [f for f in FEATURES if f not in train.columns]
print("Missing feature columns:", missing)
assert not missing

# Confirm target is constructed only after the future-month merge.
print("Feature columns:", FEATURES)
print("Label column:", "target")
print("Future outcome column kept outside X:", "future_ctr")

Missing feature columns: []
Feature columns: ['impressions', 'ctr', 'avg_position', 'engagement_rate', 'scroll_events']
Label column: target
Future outcome column kept outside X: future_ctr


## 4. Real failure examples

The examples below come from the untouched time-aware test period. They are intentionally shown with pseudonymized IDs only.

Two useful error types are reviewed:

- **False positives:** the model ranks a page highly, but the future CTR does not fall.
- **Missed positives:** the future CTR falls, but the page does not reach the model's top 50.

A failure example is not an embarrassment. It tells us what the feature set does not capture.

In [12]:
review = test[[
    "client_hash_id",
    "content_hash_id",
    "impressions",
    "ctr",
    "future_ctr",
    "avg_position",
    "engagement_rate",
    "scroll_events",
    "target"
]].copy()

review["model_score"] = time_score
review["model_rank"] = review["model_score"].rank(
    ascending=False,
    method="first"
)

top50_cutoff = review["model_score"].nlargest(
    min(50, len(review))
).min()

review["error_type"] = np.select(
    [
        (review["model_score"] >= top50_cutoff) & (review["target"] == 0),
        (review["model_score"] < top50_cutoff) & (review["target"] == 1)
    ],
    [
        "false_positive_in_top50",
        "missed_positive"
    ],
    default="other"
)

false_positives = review[
    review["error_type"] == "false_positive_in_top50"
].sort_values("model_score", ascending=False).head(5)

missed_positives = review[
    review["error_type"] == "missed_positive"
].sort_values("future_ctr", ascending=True).head(5)

print("FALSE POSITIVES")
display(false_positives)

print("MISSED POSITIVES")
display(missed_positives)

FALSE POSITIVES


,client_hash_id,content_hash_id,impressions,ctr,future_ctr,avg_position,engagement_rate,scroll_events,target,model_score,model_rank,error_type
388468,client_8ddc46da5414ffd8,content_d46321b2dce9da21,756.0,0.068783,0.072993,4.503968,NaN,NaN,0,0.847567,27201.0,false_positive_in_top50
6,client_62f4a7e64f5e0096,content_0a4f318babd3767e,2192.0,0.006843,0.010293,8.436131,NaN,NaN,0,0.847567,2.0,false_positive_in_top50
73,client_62f4a7e64f5e0096,content_a4f34119859e1da6,1705.0,0.005279,0.006295,6.123754,NaN,NaN,0,0.847567,12.0,false_positive_in_top50
83,client_62f4a7e64f5e0096,content_6084a8126e0c8d9a,50.0,0.020000,0.071429,18.740000,NaN,NaN,0,0.847567,14.0,false_positive_in_top50
95,client_62f4a7e64f5e0096,content_0d8d359d344f4cb1,2694.0,0.007424,0.007843,5.595026,NaN,NaN,0,0.847567,15.0,false_positive_in_top50


MISSED POSITIVES


,client_hash_id,content_hash_id,impressions,ctr,future_ctr,avg_position,engagement_rate,scroll_events,target,model_score,model_rank,error_type
203904,client_73cda7b4e4f265ea,content_7047e3919af1914f,3317.0,0.000301,0.0,8.196865,0.250000,1.0,1,0.458990,73430.0,missed_positive
203802,client_73cda7b4e4f265ea,content_7c7858146dcf6a0d,511.0,0.001957,0.0,7.062622,0.000000,0.0,1,0.765331,31798.0,missed_positive
203945,client_73cda7b4e4f265ea,content_76c388d85ff36e2f,1352.0,0.000740,0.0,7.088018,0.071429,5.0,1,0.546048,70674.0,missed_positive
203940,client_73cda7b4e4f265ea,content_55150652f1135a2b,456.0,0.002193,0.0,11.907895,0.000000,0.0,1,0.765331,31804.0,missed_positive
203931,client_73cda7b4e4f265ea,content_4abd3cb88a5548a7,3789.0,0.000264,0.0,6.735814,0.071429,1.0,1,0.458990,73431.0,missed_positive


### What these errors mean

A false positive can occur because low current CTR, visibility and engagement signals are not sufficient to explain every future movement. Search demand, SERP composition, seasonality, query mix, competition and other unobserved factors can change.

A missed positive can occur because the current feature set does not capture the specific condition that preceded the future CTR decline.

Therefore, the model should rank pages for human review rather than automatically prescribe a title, metadata change, content rewrite or other intervention.

## 5. Claim rewrite

### Claim that would be too strong

> "The Decision Tree predicts which pages need CTR fixes and can identify the pages Google will demote."

### Honest replacement

> "Under the tested time-aware split, the Decision Tree measured how well current-month search and engagement signals prioritized pages whose next-month CTR declined. The output is a directional review ranking and decision-support signal; it does not establish why CTR changed or predict Google's ranking decisions."

### Another claim that would be too strong

> "The model proves that improving engagement will improve search performance."

### Honest replacement

> "Current-month engagement measures were included as observed signals. Their measured association with the future CTR-drop proxy does not establish that changing engagement will cause better search performance."

### What I can safely report

- **Observed:** the model's measured Precision@50 on the defined test window.
- **Measured:** the difference between random and time-aware validation.
- **Directional:** feature importance and error patterns suggest which observed signals may be useful for prioritization.
- **Decision-support:** the ranked output can help a reviewer decide which pages to inspect first.
- **Not established:** causality, Google's ranking mechanism, or guaranteed improvement after an intervention.

## 6. Self-check

- [x] Two research-paper findings named.
- [x] A concrete methodology question is written for each finding.
- [x] Questions are constructive rather than adversarial.
- [x] Naive random validation is shown as a diagnostic.
- [x] Honest time-aware validation is shown as the preferred forward-looking evaluation.
- [x] Before/after Precision@50 comparison is printed.
- [x] Model features are audited for future leakage.
- [x] Product decision fields are excluded.
- [x] Real pseudonymized failure examples are displayed.
- [x] Claims are rewritten into observed / measured / directional / decision-support language.
- [x] No client names, domains, raw URLs, private queries, credentials or raw exports are published.

### Final reviewer statement

The strongest result of this notebook is not necessarily a higher score. It is a more defensible estimate of what the model can actually support. The time-aware result should be used for the Week-5 model's forward-looking claim, and any gap between random and time-aware performance should be treated as evidence about validation optimism rather than hidden.